# DynamoDB Demo

Chỉ 4 bước: `create()` -> `status()` -> `seed()` -> `destroy(yes=True)`.

**Trước khi chạy**
- Mở notebook từ thư mục `database/Dynamo`.
- Chạy cell install bên dưới (chỉ cần lần đầu).
- `aws configure` để có credentials.
- Muốn đổi region / tên bảng thì sửa `.env` (xem `.env.example`).

**Các lệnh**

| Lệnh | Việc |
|---|---|
| `create()` | Tạo bảng on-demand + bật PITR. Vài giây. |
| `status()` | In trạng thái, ARN, key schema, billing mode, PITR, số item. |
| `seed()` | Ghi 80 đơn trải ~3 năm theo batch 25 item, rồi đếm lại. |
| `count_items()` | Đếm chính xác bằng scan. |
| `destroy(yes=True)` | Xóa bảng khi xong. |

Dùng `count_items()` vì `ItemCount` của DynamoDB chỉ là số xấp xỉ, AWS cập nhật mỗi khoảng 6 giờ.

**Thiết kế bảng**

Mô phỏng một hệ thống legacy: chỉ có `order_id` (PK) + `created_at` (SK), **không** có GSI
hay field phục vụ archival. Vì vậy archival về sau phải dựa vào scan + filter theo `status`
và các mốc thời gian có sẵn (`closed_at`, `created_at`).

Field của item: `order_id`, `created_at`, `updated_at`, `customer_id`, `status`, `amount`,
`currency`, và `closed_at` (chỉ có ở đơn đã ở trạng thái terminal).

Phân bố data seed (80 đơn, cách nhau 14 ngày):

| Tuổi đơn | Status | `closed_at` |
|---|---|---|
| hơn 365 ngày | `CLOSED`, hoặc `CANCELLED` mỗi đơn thứ 5 | có |
| 30 đến 365 ngày | `CLOSED` | có |
| dưới 30 ngày | `OPEN` hoặc `IN_PROGRESS` | không |

**Xem data bằng GUI**

AWS Console > DynamoDB > Tables > tên bảng > Explore items.

DynamoDB gọi qua HTTPS (443) nên không bị mạng công ty chặn như RDS ở port 5432.

**Chi phí**

On-demand billing nên chỉ trả tiền khi đọc/ghi, 80 items gần như miễn phí.
Nhớ `destroy(yes=True)` khi xong để không bị tính storage.

In [ ]:
%pip install boto3 python-dotenv ipykernel -q

## Setup 1 — Config & helpers

In [ ]:
import os
import random
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

try:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(os.getcwd(), ".env"))
except ImportError:
    pass

AWS_REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
PROJECT = os.environ.get("PROJECT", "archival-demo")
DDB_TABLE = os.environ.get("DDB_TABLE", f"{PROJECT}-orders")

TOTAL_ITEMS = 80
STEP_DAYS = 14        # 80 * 14 ngày => trải ~3 năm
OLD_ORDER_DAYS = 365  # chỉ dùng để fake data trông thật, KHÔNG phải logic archival


def ddb():
    return boto3.client("dynamodb", region_name=AWS_REGION)


def table_exists(client):
    try:
        client.describe_table(TableName=DDB_TABLE)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            return False
        raise


def _iso(days_ago):
    return (datetime.now(timezone.utc) - timedelta(days=days_ago)).strftime("%Y-%m-%dT%H:%M:%SZ")


def build_items():
    """Sinh 80 order trải ~3 năm. Đơn càng cũ càng ở trạng thái terminal."""
    items, old, recent = [], 0, 0
    for i in range(TOTAL_ITEMS):
        days_ago = i * STEP_DAYS
        item = {
            "order_id": {"S": f"ORD-{i:06d}"},
            "created_at": {"S": _iso(days_ago)},
            "updated_at": {"S": _iso(days_ago - 1)},
            "customer_id": {"S": f"CUST-{(i % 50) + 1:04d}"},
            "amount": {"N": f"{random.Random(i).random() * 990 + 10:.2f}"},
            "currency": {"S": "USD"},
        }
        if days_ago > OLD_ORDER_DAYS:
            item["status"] = {"S": "CANCELLED" if i % 5 == 0 else "CLOSED"}
            item["closed_at"] = {"S": _iso(days_ago - 5)}
            old += 1
        elif days_ago > 30:
            item["status"] = {"S": "CLOSED"}
            item["closed_at"] = {"S": _iso(days_ago - 3)}
            recent += 1
        else:
            item["status"] = {"S": "IN_PROGRESS" if i % 3 == 0 else "OPEN"}
            recent += 1
        items.append(item)
    return items, old, recent


def write_batch(client, batch):
    """Ghi tối đa 25 item, retry phần UnprocessedItems."""
    request = {DDB_TABLE: [{"PutRequest": {"Item": it}} for it in batch]}
    for _ in range(5):
        resp = client.batch_write_item(RequestItems=request)
        unprocessed = resp.get("UnprocessedItems") or {}
        if not unprocessed.get(DDB_TABLE):
            return
        request = unprocessed
    raise RuntimeError("Còn item chưa ghi được sau 5 lần thử.")


print(f"setup ready: region={AWS_REGION} table={DDB_TABLE}")

## Setup 2 — create / status / seed / destroy

In [ ]:
def create():
    session = boto3.Session(region_name=AWS_REGION)
    client = session.client("dynamodb")
    account = session.client("sts").get_caller_identity()["Account"]
    print(f"AWS account {account} / {AWS_REGION}")

    if table_exists(client):
        print(f"{DDB_TABLE} đã tồn tại, bỏ qua create.")
        return

    print(f"Creating table {DDB_TABLE} ...")
    client.create_table(
        TableName=DDB_TABLE,
        BillingMode="PAY_PER_REQUEST",
        AttributeDefinitions=[
            {"AttributeName": "order_id", "AttributeType": "S"},
            {"AttributeName": "created_at", "AttributeType": "S"},
        ],
        KeySchema=[
            {"AttributeName": "order_id", "KeyType": "HASH"},
            {"AttributeName": "created_at", "KeyType": "RANGE"},
        ],
        Tags=[{"Key": "Project", "Value": PROJECT}, {"Key": "Environment", "Value": "dev"}],
    )
    print("Waiting for ACTIVE ...")
    client.get_waiter("table_exists").wait(TableName=DDB_TABLE)

    print("Enabling Point-in-Time Recovery (cần cho native export sang S3) ...")
    client.update_continuous_backups(
        TableName=DDB_TABLE,
        PointInTimeRecoverySpecification={"PointInTimeRecoveryEnabled": True},
    )
    print("Table sẵn sàng (on-demand, PITR on, không có GSI).")


def status():
    client = ddb()
    if not table_exists(client):
        print(f"{DDB_TABLE} không tồn tại. Chạy create() trước.")
        return
    t = client.describe_table(TableName=DDB_TABLE)["Table"]
    keys = {k["KeyType"]: k["AttributeName"] for k in t["KeySchema"]}
    billing = t.get("BillingModeSummary", {}).get("BillingMode", "PROVISIONED")
    try:
        pitr = client.describe_continuous_backups(TableName=DDB_TABLE)[
            "ContinuousBackupsDescription"
        ]["PointInTimeRecoveryDescription"]["PointInTimeRecoveryStatus"]
    except ClientError:
        pitr = "UNKNOWN"

    print(f"Status: {t['TableStatus']}")
    print("")
    print("--- Thông tin bảng ---")
    print(f"Table    : {t['TableName']}")
    print(f"Region   : {AWS_REGION}")
    print(f"ARN      : {t['TableArn']}")
    print(f"Keys     : PK={keys.get('HASH')}  SK={keys.get('RANGE')}")
    print(f"Billing  : {billing}")
    print(f"PITR     : {pitr}")
    print(f"Items    : ~{t.get('ItemCount', 0)} (ApproximateItemCount, AWS cập nhật mỗi ~6h)")
    print(f"Size     : {t.get('TableSizeBytes', 0)} bytes")
    print("")
    print("Đếm chính xác:  count_items()")
    print(f"Xem bằng GUI :  AWS Console > DynamoDB > Tables > {DDB_TABLE} > Explore items")


def count_items():
    """Đếm chính xác bằng scan (ApproximateItemCount bị trễ ~6h)."""
    client = ddb()
    if not table_exists(client):
        print(f"{DDB_TABLE} không tồn tại.")
        return 0
    total, key = 0, None
    while True:
        kw = {"TableName": DDB_TABLE, "Select": "COUNT"}
        if key:
            kw["ExclusiveStartKey"] = key
        resp = client.scan(**kw)
        total += resp["Count"]
        key = resp.get("LastEvaluatedKey")
        if not key:
            break
    print(f"{DDB_TABLE}: {total} items")
    return total


def seed():
    client = ddb()
    if not table_exists(client):
        raise RuntimeError(f"{DDB_TABLE} không tồn tại. Chạy create() trước.")

    items, old, recent = build_items()
    print(f"Generated {len(items)} items")
    for n in range(0, len(items), 25):
        batch = items[n:n + 25]
        write_batch(client, batch)
        print(f"  batch {n // 25 + 1} written ({len(batch)} items)")

    print("")
    print(f"Seed xong: {len(items)} items ({old} old/terminal, {recent} recent)")
    print("Data chỉ có field nghiệp vụ tự nhiên, không có field archival.")
    count_items()


def destroy(yes=False):
    if not yes:
        print("Cần gọi destroy(yes=True) để xác nhận.")
        return
    client = ddb()
    if not table_exists(client):
        print(f"{DDB_TABLE} không tồn tại, bỏ qua.")
        return
    print(f"Deleting {DDB_TABLE} ...")
    client.delete_table(TableName=DDB_TABLE)
    print("Waiting for deletion ...")
    client.get_waiter("table_not_exists").wait(TableName=DDB_TABLE)
    print("Teardown xong.")


print("commands ready: create() / status() / seed() / destroy(yes=True)")

## Bước 1 — Tạo bảng DynamoDB
On-demand billing, bật PITR. Chỉ mất vài giây.

In [ ]:
create()

## Bước 2 — Trạng thái + thông tin bảng

In [ ]:
status()

## Bước 3 — Nạp fake data
80 đơn trải ~3 năm, ghi theo batch 25 item.

In [ ]:
seed()

## Bước 4 — Xóa bảng
Xóa vĩnh viễn bảng và toàn bộ data.

In [ ]:
destroy(yes=True)